# 2-D residuals with and without the Earth-centric inputs (Nov 4, 2026)

### Summary
* Dr. Saydjari asked (Nov 3) for the 2-D residual map (notebook 10282026 §5) to be redone as **Mollweide maps**,
  comparing the model **with** and **without the Earth-centric inputs**.
* **Round 43** trained the best model (parallax band layer, cos×exp at 1000 epochs) with n·δ̂ (Earth-limb direction) and
  n·M̂ (Moon direction) added, on all 6 shifts × 5 seeds (Discovery jobs 692009–692014, 6.5–10.5 h each).
* **The two maps are the same to within ~0.006% (rms)** in both frames, and **neither shows a trailing-cloud residual.**
  As expected, the simulated Kelsall sky has no Earth- or Moon-centred light. This repeats round 33's null test on the
  current best model. These inputs only matter for real data (Earthshine, moonlight).


In [ ]:
const R = joinpath(@__DIR__, "..")
struct PNGFile; path::String; end; Base.show(io::IO, ::MIME"image/png", p::PNGFile) = write(io, read(p.path))   # raw PNG bytes: renders in any Jupyter front end
showpng(f) = PNGFile(joinpath(R, "figures", f))


## 1. The maps (`sweeps/earth2d.jl`)
Mean residual (network − Kelsall, %) of the 5-network ensembles on the withheld interpolation windows of all 6 shifts, in
3° cells. Red means the network is too bright, blue too faint.
* **Top row: Earth-motion frame.** Longitude is measured from Earth's direction of motion (0 = ahead, 90 = Sun, 180 =
  behind, 270 = away from the Sun); latitude is ecliptic latitude. The map is centred on **behind Earth**, where Kelsall's
  trailing cloud sits. SPHEREx always looks ~90° from the Sun, so it covers only the band ahead of Earth (split across the
  two edges) and the band behind it (centre). The Sun and anti-Sun directions are never observed (grey).
* **Bottom row: ecliptic frame**, centred on ecliptic longitude 0°, like our other sky maps. The whole sky is covered.
* **Columns:** without the Earth-centric inputs (cos×exp 1000, rounds 37–38), with them (round 43), and with − without.


In [ ]:
showpng("11042026_earth2d_moll.png")


## 2. Numbers

| | without Earth-centric inputs | with (n·δ̂, n·M̂) |
|---|---|---|
| ensemble test MSE, shifts 0/10/20/30/40/50 (×10⁻⁸) | 1.38 1.46 1.70 1.57 1.50 1.53 | 1.40 1.54 1.68 1.56 1.59 1.56 |
| mean over shifts | **1.52** | 1.56 |
| sky map: median / p95 | 0.0130% / 0.038% | 0.0130% / **0.037%** |
| within 2° of the ecliptic | 0.0181% | **0.0173%** |
| missing share of the trailing cloud, \|β\| < 15° | −0.0000 ± 0.0001 | −0.0001 ± 0.0001 |
| missing share of the trailing cloud, whole sky | +0.0002 ± 0.0001 | +0.0000 ± 0.0001 |
| mean residual behind Earth (φ 150–210°, \|β\| < 15°) | +0.0004% | +0.0001% |
| mean residual ahead of Earth (φ 330–30°, \|β\| < 15°) | +0.0007% | +0.0014% |

with − without, per exposure: mean +0.0003%, rms 0.006% (half the typical error, 0.013%).

* **No trailing-cloud residual in either model.** The missing share is the fitted slope of residual against the cloud's
  share of the light. −1 would mean the model misses the whole cloud, −0.01 would mean it misses 1% of it. We get
  0.0000–0.0002 ± 0.0001, so the network reproduces the trailing cloud to better than ~0.02% of it, with or without the
  Earth-centric inputs.
* **The Earth-centric inputs neither help nor hurt.** The mean MSE is 3% higher with them (1.56 vs 1.52), and the ecliptic
  is 4% better (0.0173% vs 0.0181%). Both are within the usual seed-to-seed spread, and the difference maps are almost
  white. The only visible structure is a few cells at the inner edge of the ahead band (±0.01–0.02%), at high latitude. The behind/ahead residuals differ by ~0.001%, 10× below the typical error.
* So on simulated data the network finds no spurious Earth- or Moon-related signal, consistent with round 33. On real
  SPHEREx data, where Earthshine and moonlight can leak into the images, these are the inputs that would pick them up.


## 3. Positive control: inject a known Earth/Moon signal (rounds 44 / 44b, `sweeps/report44.jl`)
A null result only means something if the test *can* find a signal. So we added a fake **Earthshine** (gE, bright where
the line of sight points along n·δ̂, the Earth–spacecraft direction) and a fake **moonlight** (gM = exp(−θMoon/30°)) to
the training target, at A = 1% and 0.1% of the mean sky, then trained three models (shift 0, seeds 42–44):
* **base**, the best model as is (no Earth-centric inputs);
* **with n·δ̂, n·M̂**, the round-43 model;
* **base without n·S**. n·δ̂ = (n·S − n·E)/|δ| can be rebuilt from the base inputs (R² 0.9998), so we removed n·S to try
  to cut that leak.

On the withheld months, we fit network − clean Kelsall = cE·(injected Earthshine) + cM·(injected moonlight) + c0. A fit of
1 means the signal is fully learned, 0 means it is missed.

| A | model | cE | cM | rms left over (% of y) |
|---|---|---|---|---|
| 1% | base | 0.98 | −0.16 | 0.159 (injected 0.364) |
| 1% | with n·δ̂, n·M̂ | **1.00** | **1.00** | **0.021** |
| 1% | base without n·S | 0.97 | −0.09 | 0.152 |
| 0.1% | base | 0.91 | −0.06 | 0.024 (injected 0.036) |
| 0.1% | with n·δ̂, n·M̂ | 1.04 | 1.02 | 0.021 |
| 0.1% | base without n·S | 0.90 | −0.06 | 0.024 |


In [ ]:
showpng("11042026_r44_injection.png")


* **The test works.** When there is a real Earth/Moon signal, the model with the Earth-centric inputs learns all of it
  (test MSE 40–60× lower than base at 1%). Round 43's null therefore means the simulated sky has no such signal.
* **Moonlight can only be learned with n·M̂.** Without it, the moonlight is missed (cM ≈ 0) and the leftover error is
  7× larger.
* **Earthshine is learned ~90–100% even without n·δ̂.** It is still learned with n·S removed, so the leak doesn't run only
  through n·S: the other inputs and the band layer recover the Earth direction anyway.
* The figure plots the sum of both signals, so the base models still follow the line roughly (their Earthshine is
  learned). The fitted cE/cM in the table separate the two.


## 4. The trailing cloud on and off (round 45, `sweeps/report45.jl`)
The mentor's Week 13 request. The best model (parallax band layer, cos×exp 1000, 6 shifts × 5 seeds) is trained twice:
* **cloud ON**: the full Kelsall sky (rounds 37–38);
* **cloud OFF**: the same sky with Kelsall's Earth-trailing cloud taken out (`ZODI_NOCLOUD=1`; the emission file stores the
  cloud, median 2.4% of the sky, max 9%).

Each family is scored against its own target. The difference of the two ensembles' predictions, ON − OFF, is what the
networks think the cloud is, and it can be compared directly with Kelsall's cloud.


In [ ]:
showpng("11042026_r45_cloud_onoff.png")


| | cloud ON | cloud OFF |
|---|---|---|
| ensemble test MSE, shifts 0/10/20/30/40/50 (×10⁻⁸) | 1.38 1.46 1.70 1.57 1.50 1.53 | 1.28 1.41 1.73 1.47 1.47 1.42 |
| mean over shifts | 1.52 | **1.46** |
| sky map: median / p95 | 0.0130% / 0.038% | 0.0129% / 0.037% |
| within 2° of the ecliptic | 0.0181% | 0.0183% |

ON − OFF compared with Kelsall's cloud (fit ON − OFF = slope · cloud + intercept, both as % of the sky):

| | slope | intercept | rms of (ON − OFF − cloud) | rms of the cloud |
|---|---|---|---|---|
| \|β\| < 15° | 1.0000 | −0.00003% | 0.012% | 5.0% |
| whole sky | 1.0001 | −0.0003% | 0.006% | 3.6% |

* **The cloud costs the model almost nothing.** Without it, the test MSE is 4% lower (1.46 vs 1.52), within the usual
  seed-to-seed spread. The sky maps and the ecliptic error are the same.
* **The networks learn the cloud exactly.** ON − OFF reproduces Kelsall's cloud with slope 1.0000. What is left over is
  0.006–0.012% of the sky, which is about 0.2% of the cloud itself and the same size as the model's normal error.
* The leftover map (last column) is white, apart from a few cells on the ecliptic. Those are where the ordinary model
  errors of the two families don't cancel, not a cloud-shaped pattern.
* So on simulated data the model absorbs the cloud fully, without being told it exists. That suggests it could also learn
  an Earth-trailing structure on real data that differs from Kelsall's, as long as the inputs describe it (Earth's orbital
  phase and position).
